# Forge Layer (Silver) - Cleaning & Logical Transformations

## Overview
This notebook performs **data quality cleaning and logical transformations** on intake (bronze) tables.

**Silver Layer Philosophy:**
* Clean and standardize data (types, formats, nulls)
* Add logical calculations and enrichments
* Validate and flag quality issues
* Keep source table structure (no dimensional modeling yet)
* **Dimensional modeling (dim/fact) happens in Gold layer**

## Data Inventory from Intake Layer

### Master Data (Batch 1)
* `workspace.intake.driver_master` - Driver profiles and assignments
* `workspace.intake.vehicle_master` - Vehicle specifications
* `workspace.intake.route_master` - Route definitions
* `workspace.intake.trip_master` - Trip transactions

### Telemetry & IoT (Batch 2)
* `workspace.intake.core_telemetry` - 36 vehicle sensor metrics
* `workspace.intake.core_engine` - Engine-specific metrics
* `workspace.intake.gps` - GPS tracking data
* `workspace.intake.driver_behavior` - Behavioral analytics

### Operational Data (Batch 3)
* `workspace.intake.fuel_transactions` - Fuel purchase records
* `workspace.intake.insurance_claims` - Claim submissions
* `workspace.intake.maintenance` - Service records
* `workspace.intake.weather` - Weather conditions

### Document Data (Batch 4)
* `workspace.intake.accident_reports_batch4` - PDF reports
* `workspace.intake.insurance_claims_batch4` - PDF claims
* `workspace.intake.metadata_batch4_files` - Document metadata

## Silver Layer Transformations (Cleaning + Logical)

### 1. Data Type Standardization
* Fix GPS timestamp (STRING → TIMESTAMP)
* Ensure numeric columns are proper types (INT, DOUBLE, BIGINT)
* Standardize date formats across all tables
* Cast boolean flags properly

### 2. Data Quality & Validation
* **Remove duplicates** based on business keys
* **Handle nulls**: Set defaults or flag for business review
* **Validate ranges**: Speed limits, temperatures, fuel levels
* **Add quality flags**: `is_valid_record`, `has_data_issues`
* **Trim whitespace** from string columns

### 3. Logical Calculations (Derived Columns)
* **Trip duration** from start/end timestamps
* **Speeding violations** flag (speed > speed_limit)
* **Fuel consumption** calculations (fuel_level changes)
* **Idle time percentage** from telemetry
* **Distance validation** (GPS vs reported distance)

### 4. Data Enrichment (Preserve Source Schema)
* Add calculated fields to existing tables (no new tables)
* Add metadata: `processing_timestamp`, `source_batch`
* Flatten nested JSON if needed (keep column structure simple)
* Standardize column naming (snake_case)

### 5. Data Completeness
* Flag incomplete records (missing critical fields)
* Add record counts and checksums for reconciliation
* Track data lineage metadata

## Proposed Silver Layer Tables

**1:1 mapping from Intake → Forge** (same structure, cleaned data)

### Master Data Tables (Batch 1)
1. **forge.driver_master** - Cleaned driver profiles + calculated risk flags
2. **forge.vehicle_master** - Cleaned vehicle specs + age/usage metrics
3. **forge.route_master** - Cleaned route definitions + distance validation
4. **forge.trip_master** - Cleaned trips + trip_duration, trip_date, validation flags

### Telemetry Tables (Batch 2)
5. **forge.core_telemetry** - Cleaned sensor data + anomaly flags, derived metrics
6. **forge.core_engine** - Cleaned engine metrics + performance indicators
7. **forge.gps** - Fixed timestamp, speeding flags, location validation
8. **forge.driver_behavior** - Cleaned behavior data + risk scoring

### Operational Tables (Batch 3)
9. **forge.fuel_transactions** - Cleaned fuel data + consumption calculations
10. **forge.insurance_claims** - Cleaned claims + amount validation
11. **forge.maintenance** - Cleaned maintenance + cost/frequency metrics
12. **forge.weather** - Cleaned weather + condition categorization

### Document Tables (Batch 4)
13. **forge.accident_reports_batch4** - PDF metadata + extraction status
14. **forge.insurance_claims_batch4** - PDF metadata + extraction status
15. **forge.metadata_batch4_files** - Cleaned file metadata

**Note:** Dimensional modeling (star schema) will happen in the Gold layer

## Implementation Examples

### Priority 1: Critical Data Quality Fixes
Start with these high-impact transformations:

In [0]:
# Clean GPS table: fix timestamp + add logical flags
from pyspark.sql.functions import to_timestamp, col, current_timestamp, when, lit

forge_gps = (
    spark.table("workspace.intake.gps")
    # Select all original columns to preserve source schema
    .select("*")
    # Fix data type
    .withColumn("timestamp", to_timestamp(col("timestamp")))
    
    # Add logical flags
    .withColumn("is_speeding", 
                when(col("speed") > col("speed_limit"), "over speed")
                .otherwise(" limited speed"))
    .withColumn("speed_over_limit", 
                when(col("speed") > col("speed_limit"), 
                     col("speed") - col("speed_limit")).otherwise(0))
    
    # Add metadata
    .withColumn("processing_timestamp", current_timestamp())
    .withColumn("source_batch", lit("Batch2"))
    
    # Data quality flags
    .withColumn("is_valid_location", 
                (col("latitude").between(-90, 90)) & 
                (col("longitude").between(-180, 180)))
)

#display(forge_gps)
forge_gps.write.format("delta").mode("overwrite").saveAsTable("workspace.forge.gps")

In [0]:
%sql
--learning more about gps data 
--found a problem in timestamp columns datatype
--use catalog `workspace`;
--desc `intake`.`gps`;
--select * from `workspace`.`intake`.`gps` limit 100;


In [0]:
# Clean trip_master: add calculated columns + validation
from pyspark.sql.functions import lit, round as spark_round, when, hour, abs as spark_abs

forge_trip_master = (
    spark.table("workspace.intake.trip_master")
    
    # Calculate derived metrics
    .withColumn("trip_duration_minutes", 
                spark_round((col("end_time").cast("long") - col("start_time").cast("long")) / 60, 2))
    .withColumn("trip_date", col("start_time").cast("date"))
    .withColumn("trip_hour", hour(col("start_time")))
    
    # Calculate average speed validation (distance / duration)
    .withColumn("calculated_avg_speed", 
                when(col("trip_duration_minutes") > 0,
                     spark_round((col("distance_km") / col("trip_duration_minutes")) * 60, 2))
                .otherwise(0))
    
    # Data quality flags
    .withColumn("is_valid_duration", col("trip_duration_minutes") > 0)
    .withColumn("is_speed_mismatch", 
                spark_abs(col("calculated_avg_speed") - col("average_speed_kmph")) > 10)
    .withColumn("is_complete", 
                col("vehicle_id").isNotNull() & 
                col("driver_id").isNotNull() & 
                col("start_time").isNotNull() & 
                col("end_time").isNotNull())
    
    # Add metadata
    .withColumn("processing_timestamp", current_timestamp())
    .withColumn("source_batch", lit("Batch1"))
)

# display(forge_trip_master.limit(5))
# forge_trip_master.write.format("delta").mode("overwrite").saveAsTable("workspace.forge.trip_master")

In [0]:
# Clean driver_master: standardize + add risk indicators
from pyspark.sql.functions import lit, when, trim, upper, datediff, current_timestamp, round as spark_round

forge_driver_master = (
    spark.table("workspace.intake.driver_master")
    
    # Clean string fields
    .withColumn("driver_name", trim(col("driver_name")))
    .withColumn("license_number", upper(trim(col("license_number"))))
    .withColumn("license_type", upper(trim(col("license_type"))))
    
    # Calculate tenure
    .withColumn("days_with_company", 
                datediff(current_timestamp(), col("joining_date")))
    .withColumn("years_with_company", 
                spark_round(col("days_with_company") / 365, 1))
    
    # Add risk scoring flags
    .withColumn("is_high_risk", 
                col("risk_profile").isin(["High", "Very High"]))
    .withColumn("is_experienced", col("experience_years") >= 5)
    .withColumn("is_new_driver", col("days_with_company") < 90)
    
    # Data completeness flags
    .withColumn("is_complete", 
                col("driver_id").isNotNull() & 
                col("license_number").isNotNull() & 
                col("assigned_vehicle_id").isNotNull())
    
    # Add metadata
    .withColumn("processing_timestamp", current_timestamp())
    .withColumn("source_batch", lit("Batch1"))
)

# display(forge_driver_master.limit(5))
# forge_driver_master.write.format("delta").mode("overwrite").saveAsTable("workspace.forge.driver_master")

In [0]:
# Clean telemetry: keep detail records + add anomaly flags
# Note: Trip-level aggregations can be done here or in Gold layer
from pyspark.sql.functions import avg, max, min, stddev, count, sum as spark_sum

agg_telemetry_by_trip = (
    spark.table("workspace.intake.core_telemetry")
    .groupBy("trip_id", "vehicle_id", "driver_id")
    .agg(
        avg("engine_rpm").alias("avg_engine_rpm"),
        max("engine_rpm").alias("max_engine_rpm"),
        avg("fuel_efficiency").alias("avg_fuel_efficiency"),
        min("fuel_level").alias("min_fuel_level"),
        max("fuel_level").alias("max_fuel_level"),
        avg("engine_temperature").alias("avg_engine_temp"),
        max("engine_temperature").alias("max_engine_temp"),
        stddev("engine_load").alias("engine_load_variability"),
        spark_sum("idle_time").alias("total_idle_time"),
        # Count anomalies
        spark_sum(when(col("check_engine_light") == True, 1).otherwise(0)).alias("check_engine_events"),
        count("*").alias("telemetry_record_count")
    )
)

# agg_telemetry_by_trip.write.format("delta").mode("overwrite").saveAsTable("axiogo.forge.agg_trip_telemetry")

## Data Quality & Optimization Best Practices

### Partitioning Strategy
* **Transaction tables**: Partition by date (trip_date, transaction_date)
* **Large telemetry/GPS**: Partition by date (+ vehicle_id if very large)
* **Master tables**: Usually no partitioning (smaller, lookup tables)

### Z-Ordering
* Apply Z-ORDER on frequently filtered columns:
  * `OPTIMIZE workspace.forge.trip_master ZORDER BY (driver_id, vehicle_id, trip_date)`
  * `OPTIMIZE workspace.forge.gps ZORDER BY (trip_id, timestamp)`
  * `OPTIMIZE workspace.forge.core_telemetry ZORDER BY (trip_id, vehicle_id)`

### Data Quality Checks
* Row count reconciliation (intake vs forge)
* Null checks on critical fields
* Duplicate detection
* Range validation (speeds, temperatures, dates)
* Referential integrity (all trip_master.driver_id exist in driver_master)

In [0]:
# Example 5: Data Quality Validation Framework
from pyspark.sql.functions import count, sum as spark_sum, when, col, current_timestamp

def run_dq_checks(table_name, primary_key, critical_columns):
    """
    Run data quality checks on a silver layer table
    """
    df = spark.table(table_name)
    
    dq_results = {
        "table_name": table_name,
        "check_timestamp": datetime.now(),
        "total_rows": df.count(),
        "duplicate_count": df.count() - df.dropDuplicates([primary_key]).count(),
    }
    
    # Check for nulls in critical columns
    for col_name in critical_columns:
        null_count = df.filter(col(col_name).isNull()).count()
        dq_results[f"{col_name}_null_count"] = null_count
        dq_results[f"{col_name}_null_pct"] = (null_count / dq_results["total_rows"] * 100) if dq_results["total_rows"] > 0 else 0
    
    return dq_results

# Example usage
# trip_dq = run_dq_checks(
#     "axiogo.forge.fact_trip",
#     "trip_id",
#     ["vehicle_id", "driver_id", "start_time", "end_time"]
# )
# print(trip_dq)

## Recommended Implementation Approach

### Phase 1: Critical Data Quality (Week 1)
1. Create `workspace.forge` schema
2. Clean master tables: driver_master, vehicle_master, route_master
3. Fix GPS timestamp issue → forge.gps
4. Clean trip_master with calculated columns → forge.trip_master

### Phase 2: Telemetry & IoT (Week 2)
5. Clean core_telemetry with anomaly flags
6. Clean core_engine metrics
7. Clean driver_behavior data
8. Add data quality framework

### Phase 3: Operational Data (Week 3)
9. Clean fuel_transactions with calculations
10. Clean maintenance records
11. Clean insurance_claims
12. Clean weather data

### Phase 4: Documents & Validation (Week 4)
13. Process PDF metadata tables
14. Implement comprehensive data quality checks
15. Add row count reconciliation
16. Set up data quality monitoring

### Phase 5: Optimization (Ongoing)
17. Apply partitioning strategies
18. Implement Z-ordering
19. Set up incremental refresh patterns (MERGE for updates)
20. Performance tuning

---

**Key Architectural Principles:**
* **1:1 table mapping** from intake → forge (preserve source structure)
* **Cleaning focus**: Fix types, validate data, add flags
* **Logical transformations**: Calculate derived columns in-place
* **No dimensional modeling** in Silver (save for Gold layer)
* **Delta Lake** for all tables (ACID, time travel)
* **Incremental processing** with MERGE for updates
* **Unity Catalog** for governance